# Step 6 — Enhancer gate ablation (Step B)

Step A (offline score fusion) failed → ask whether **Wave-U-Net itself** can help ECAPA under noise.

Notebook 03’s P3 uses `enhancer.process()` which **blends** noisy + enhanced via learned `gate_net`.
This smoke compares waveform gate modes on **dev**, SNR **5 / 0** only:

| ID | Waveform |
|----|----------|
| **B0** | noisy (no enhance) |
| **enh_learned** | current `gate_net` blend (= old P3 path) |
| **enh_full** | ungated enhanced only (`g=1`) |
| **enh_force0** | force `g=0` (sanity ≈ B0) |

**Pass bar:** `enh_full` SASV-EER **&lt; B0** on **both** SNR 5 and 0.

- Pass → rebuild P2 using ungated enhance (worth a fuller run).
- Fail → this checkpoint is a poor fit for SASV+MUSAN; stop enhancer chasing.

Defaults: `MAX_TRIALS=1500` smoke (not full 29k). Use CUDA kernel.


In [4]:
from pathlib import Path
import sys
import json
import numpy as np
import pandas as pd
import torch
from tqdm.auto import tqdm

ROOT = Path.cwd().resolve()
if not (ROOT / "noise_gated_lib.py").exists():
    ROOT = ROOT / "replay-cnn-baseline" / "experiments" / "sasv_noise_gated"
sys.path.insert(0, str(ROOT))
sys.path.insert(0, str(ROOT.parent / "sasv_la2019"))

from noise_gated_lib import (
    DEFAULT_LA,
    DEFAULT_SASV,
    RUNS_DIR,
    build_speaker_models,
    cosine,
    eers_from_preds,
    embed_waveform,
    ensure_dirs,
    ensure_sasv_on_path,
    ensure_server_on_path,
    load_app_ecapa,
    load_enhancer,
    load_waveform,
    maybe_noise_waveform,
    patch_speechbrain_windows_lazy_import,
    read_trials,
    resolve_audio_path,
    resolve_noise_bank,
    save_json,
    si_sdr,
    snr_tag,
    trial_key_counts,
    waveunet_process,
    write_score_csv,
)

ensure_dirs()
ensure_server_on_path()
patch_speechbrain_windows_lazy_import()

SPLIT = "dev"
MAX_TRIALS = 1500  # smoke; set 0 for full (slow)
SNRS_DB = (5, 0)
SI_SDR_PROBE = 50  # waveforms for quality probe per SNR
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
SEED = 20260926
RNG = np.random.default_rng(SEED)

NOISE_ROOT = Path(r"D:/downloads/musan/musan/noise")
noise_bank, noise_used = resolve_noise_bank(NOISE_ROOT)

OUT = RUNS_DIR / "step6_enhancer_gate_ablation"
OUT.mkdir(parents=True, exist_ok=True)

print("device", DEVICE, "split", SPLIT, "max_trials", MAX_TRIALS)
print("SNRs", [snr_tag(s) for s in SNRS_DB])
print("MUSAN", noise_used, "clips", len(noise_bank))
print("out", OUT)


[noise] using 62 clips from D:\downloads\musan\musan\noise
device cuda split dev max_trials 1500
SNRs ['snr5db', 'snr0db']
MUSAN D:\downloads\musan\musan\noise clips 62
out D:\speaker-verification-system\replay-cnn-baseline\experiments\sasv_noise_gated\runs\step6_enhancer_gate_ablation


### Load ECAPA + Wave-U-Net + enrol


In [5]:
sasv = ensure_sasv_on_path(DEFAULT_SASV)
trials = read_trials(sasv, SPLIT, max_trials=MAX_TRIALS)
print(trial_key_counts(trials))

classifier = load_app_ecapa(device=DEVICE)
enhancer = load_enhancer(device=DEVICE)
assert enhancer is not None, "Wave-U-Net failed to load — check checkpoint + denoisers"

spk_models = build_speaker_models(classifier, DEFAULT_LA, SPLIT, DEVICE)
print("speakers", len(spk_models))


{'target': 75, 'nontarget': 293, 'spoof': 1132, 'total': 1500}


Could not parse CUDA device string 'cuda': not enough values to unpack (expected 2, got 1). Falling back to device 0.


[noise_gated_lib] enhancer ok: WaveUNetEnhancer ckpt=D:\speaker-verification-system\app\server\checkpoints\waveunet_finetuned_v4_best.pt


Enrol dev:   0%|          | 0/10 [00:00<?, ?it/s]

speakers 10


### SI-SDR probe (clean reference available — we inject noise)


In [6]:
@torch.inference_mode()
def probe_si_sdr(trials, snr_db, n_probe=SI_SDR_PROBE):
    rows = []
    gate_vals = []
    for trial in tqdm(trials[:n_probe], desc=f"sisdr:{snr_tag(snr_db)}"):
        path = resolve_audio_path(DEFAULT_LA, SPLIT, trial.test_utt)
        clean = load_waveform(path)
        # fresh RNG stream per utt for reproducibility across modes
        rng = np.random.default_rng(SEED + hash((trial.test_utt, snr_db)) % (2**31))
        noisy = maybe_noise_waveform(clean, snr_db=snr_db, noise_bank=noise_bank, rng=rng)
        clean_np = clean.detach().cpu().numpy().reshape(-1)
        noisy_np = noisy.detach().cpu().numpy().reshape(-1)

        enh_l, g = waveunet_process(enhancer, noisy, gate_mode="learned")
        enh_f, _ = waveunet_process(enhancer, noisy, gate_mode="full_enhance")
        enh_0, _ = waveunet_process(enhancer, noisy, gate_mode="force_0")
        gate_vals.append(g)

        rows.append(
            {
                "snr": snr_tag(snr_db),
                "test_utt": trial.test_utt,
                "gate_learned": g,
                "sisdr_noisy": si_sdr(noisy_np, clean_np),
                "sisdr_learned": si_sdr(enh_l.numpy(), clean_np),
                "sisdr_full": si_sdr(enh_f.numpy(), clean_np),
                "sisdr_force0": si_sdr(enh_0.numpy(), clean_np),
            }
        )
    return pd.DataFrame(rows), float(np.nanmean(gate_vals)) if gate_vals else float("nan")


sisdr_frames = []
gate_means = {}
for snr_db in SNRS_DB:
    df_s, gmean = probe_si_sdr(trials, snr_db)
    sisdr_frames.append(df_s)
    gate_means[snr_tag(snr_db)] = gmean
    print(
        snr_tag(snr_db),
        "mean gate",
        round(gmean, 3),
        "SI-SDR mean",
        df_s[["sisdr_noisy", "sisdr_learned", "sisdr_full", "sisdr_force0"]].mean().round(2).to_dict(),
    )

sisdr_df = pd.concat(sisdr_frames, ignore_index=True)
sisdr_path = OUT / "si_sdr_probe.csv"
sisdr_df.to_csv(sisdr_path, index=False)
print("wrote", sisdr_path)
sisdr_df.groupby("snr")[["sisdr_noisy", "sisdr_learned", "sisdr_full", "sisdr_force0"]].mean()


sisdr:snr5db:   0%|          | 0/50 [00:00<?, ?it/s]

snr5db mean gate 0.598 SI-SDR mean {'sisdr_noisy': 5.0, 'sisdr_learned': 4.2, 'sisdr_full': 2.65, 'sisdr_force0': 5.0}


sisdr:snr0db:   0%|          | 0/50 [00:00<?, ?it/s]

snr0db mean gate 0.668 SI-SDR mean {'sisdr_noisy': 0.02, 'sisdr_learned': 0.06, 'sisdr_full': -2.84, 'sisdr_force0': 0.02}
wrote D:\speaker-verification-system\replay-cnn-baseline\experiments\sasv_noise_gated\runs\step6_enhancer_gate_ablation\si_sdr_probe.csv


,sisdr_noisy,sisdr_learned,sisdr_full,sisdr_force0
snr,,,,
snr0db,0.019951,0.055172,-2.844738,0.019951
snr5db,4.998705,4.204951,2.645121,4.998705


### Score B0 / enh_learned / enh_full / enh_force0


In [7]:
SYSTEMS = ("B0", "enh_learned", "enh_full", "enh_force0")
GATE_FOR = {
    "enh_learned": "learned",
    "enh_full": "full_enhance",
    "enh_force0": "force_0",
}


@torch.inference_mode()
def score_ablation(trials, snr_db):
    preds = {name: [] for name in SYSTEMS}
    keys = []
    rows = {name: [] for name in SYSTEMS}
    tag = snr_tag(snr_db)

    for trial in tqdm(trials, desc=f"score:{tag}"):
        spk = spk_models[trial.speaker_id]
        path = resolve_audio_path(DEFAULT_LA, SPLIT, trial.test_utt)
        clean = load_waveform(path)
        rng = np.random.default_rng(SEED + hash((trial.test_utt, snr_db)) % (2**31))
        noisy = maybe_noise_waveform(clean, snr_db=snr_db, noise_bank=noise_bank, rng=rng)

        emb_b0 = embed_waveform(classifier, noisy, DEVICE)
        s_b0 = float(cosine(spk, emb_b0))
        preds["B0"].append(s_b0)
        rows["B0"].append(
            {
                "speaker_id": trial.speaker_id,
                "test_utt": trial.test_utt,
                "key": trial.key,
                "snr": tag,
                "score": s_b0,
            }
        )

        for name, mode in GATE_FOR.items():
            enh_wave, g = waveunet_process(enhancer, noisy, gate_mode=mode)
            emb = embed_waveform(classifier, enh_wave.float(), DEVICE)
            s = float(cosine(spk, emb))
            preds[name].append(s)
            rows[name].append(
                {
                    "speaker_id": trial.speaker_id,
                    "test_utt": trial.test_utt,
                    "key": trial.key,
                    "snr": tag,
                    "score": s,
                    "gate": g,
                }
            )

        keys.append(trial.key)

    return rows, preds, keys


all_summaries = {}
metric_rows = []

for snr_db in SNRS_DB:
    tag = snr_tag(snr_db)
    rows, preds, keys = score_ablation(trials, snr_db)
    out = OUT / f"{SPLIT}_{tag}"
    out.mkdir(parents=True, exist_ok=True)

    summaries = {}
    for name in SYSTEMS:
        fields = ["speaker_id", "test_utt", "key", "snr", "score"]
        if name != "B0":
            fields.append("gate")
        write_score_csv(out / f"{name}_scores.csv", rows[name], fields)
        summaries[name] = eers_from_preds(preds[name], keys)
        metric_rows.append(
            {
                "snr": tag,
                "system": name,
                "sasv_eer_percent": summaries[name]["sasv_eer_percent"],
                "sv_eer_percent": summaries[name]["sv_eer_percent"],
                "spf_eer_percent": summaries[name]["spf_eer_percent"],
                "n_trials": len(keys),
            }
        )

    save_json(out / "metrics.json", summaries)
    all_summaries[tag] = summaries
    print(tag, {k: round(v["sasv_eer_percent"], 3) for k, v in summaries.items()})

metrics_df = pd.DataFrame(metric_rows)
display(metrics_df)
metrics_df.to_csv(OUT / "metrics_long.csv", index=False)
pivot = metrics_df.pivot_table(index="system", columns="snr", values="sasv_eer_percent")
display(pivot.round(3))
pivot.to_csv(OUT / "matrix_sasv_eer.csv")
save_json(OUT / "all_snrs.json", all_summaries)


score:snr5db:   0%|          | 0/1500 [00:00<?, ?it/s]

snr5db {'B0': 15.298, 'enh_learned': 23.789, 'enh_full': 22.667, 'enh_force0': 15.298}


score:snr0db:   0%|          | 0/1500 [00:00<?, ?it/s]

snr0db {'B0': 17.614, 'enh_learned': 20.0, 'enh_full': 24.0, 'enh_force0': 17.614}


,snr,system,sasv_eer_percent,sv_eer_percent,spf_eer_percent,n_trials
0,snr5db,B0,15.298246,4.000000,18.727915,1500
1,snr5db,enh_learned,23.789474,9.333333,26.666667,1500
2,snr5db,enh_full,22.666667,8.000000,26.590106,1500
3,snr5db,enh_force0,15.298246,4.000000,18.727915,1500
4,snr0db,B0,17.614035,5.333333,21.113074,1500
5,snr0db,enh_learned,20.000000,9.556314,22.666667,1500
6,snr0db,enh_full,24.000000,7.849829,26.666667,1500
7,snr0db,enh_force0,17.614035,5.333333,21.113074,1500


snr,snr0db,snr5db
system,,
B0,17.614,15.298
enh_force0,17.614,15.298
enh_full,24.000,22.667
enh_learned,20.000,23.789


### Step B decision


In [8]:
b0 = metrics_df[metrics_df.system == "B0"].set_index("snr")["sasv_eer_percent"]
full = metrics_df[metrics_df.system == "enh_full"].set_index("snr")["sasv_eer_percent"]
learned = metrics_df[metrics_df.system == "enh_learned"].set_index("snr")["sasv_eer_percent"]
force0 = metrics_df[metrics_df.system == "enh_force0"].set_index("snr")["sasv_eer_percent"]

tags = [snr_tag(s) for s in SNRS_DB]
deltas_full = {t: float(full[t] - b0[t]) for t in tags}
deltas_learned = {t: float(learned[t] - b0[t]) for t in tags}
pass_bar = all(deltas_full[t] < 0 for t in tags)

# sanity: force0 should be close to B0
force0_ok = all(abs(float(force0[t] - b0[t])) < 1.0 for t in tags)

sisdr_means = (
    sisdr_df.groupby("snr")[["sisdr_noisy", "sisdr_learned", "sisdr_full", "sisdr_force0"]]
    .mean()
    .round(3)
    .to_dict()
)

decision = {
    "step": "B_enhancer_gate_ablation",
    "split": SPLIT,
    "max_trials": MAX_TRIALS,
    "snrs": tags,
    "device": DEVICE,
    "pass_criterion": "enh_full SASV-EER < B0 on snr5 and snr0",
    "pass": bool(pass_bar),
    "delta_enh_full_minus_b0": deltas_full,
    "delta_enh_learned_minus_b0": deltas_learned,
    "force0_approx_b0": bool(force0_ok),
    "mean_learned_gate": gate_means,
    "si_sdr_means": sisdr_means,
    "recommendation": (
        "Pass: rebuild P2 with ungated (full) enhance + score/emb fusion; then fuller SNR grid."
        if pass_bar
        else "Fail: Wave-U-Net (even ungated) does not help ECAPA on this smoke — "
        "do not burn a full 03 re-run; accept negative result or change enhancer/CM."
    ),
}
save_json(OUT / "step_b_decision.json", decision)
print(json.dumps(decision, indent=2))


{
  "step": "B_enhancer_gate_ablation",
  "split": "dev",
  "max_trials": 1500,
  "snrs": [
    "snr5db",
    "snr0db"
  ],
  "device": "cuda",
  "pass_criterion": "enh_full SASV-EER < B0 on snr5 and snr0",
  "pass": false,
  "delta_enh_full_minus_b0": {
    "snr5db": 7.368421052592948,
    "snr0db": 6.385964912332863
  },
  "delta_enh_learned_minus_b0": {
    "snr5db": 8.49122807002818,
    "snr0db": 2.3859649123413327
  },
  "force0_approx_b0": true,
  "mean_learned_gate": {
    "snr5db": 0.597729144692421,
    "snr0db": 0.6683938813209533
  },
  "si_sdr_means": {
    "sisdr_noisy": {
      "snr0db": 0.02,
      "snr5db": 4.999
    },
    "sisdr_learned": {
      "snr0db": 0.055,
      "snr5db": 4.205
    },
    "sisdr_full": {
      "snr0db": -2.845,
      "snr5db": 2.645
    },
    "sisdr_force0": {
      "snr0db": 0.02,
      "snr5db": 4.999
    }
  },
  "recommendation": "Fail: Wave-U-Net (even ungated) does not help ECAPA on this smoke \u2014 do not burn a full 03 re-run; accept

### Done

Outputs under `runs/step6_enhancer_gate_ablation/`.

Expect ~minutes–hours depending on GPU and `MAX_TRIALS` (1500 × 2 SNRs × 4 paths).
